# 🗃️ Bases de datos con SQLite
*(introducción práctica — versión explicada paso a paso)*

---

En este notebook aprenderás a crear y usar tu primera base de datos, usando **SQLite** — que viene incluida con Python en el módulo `sqlite3`. Toda la BBDD será un **único fichero** `.db` en tu disco, sin necesidad de instalar ningún servidor.

Al final tendrás:

* Una tabla `Estudiantes` con varios registros.
* Las herramientas para insertar, consultar, filtrar y agregar datos.
* La comprensión del patrón fundamental **conectar → cursor → SQL → commit → cerrar**.

## 1. El patrón de 5 pasos

**Toda** interacción con SQLite sigue este patrón. Grábalo a fuego:

```python
import sqlite3

conexion = sqlite3.connect('mi.db')   # 1) conectar
cursor = conexion.cursor()            # 2) obtener cursor

cursor.execute('SQL aquí...')          # 3) ejecutar SQL

conexion.commit()                      # 4) confirmar (SOLO si modificas)
conexion.close()                       # 5) cerrar
```

* Paso 1: `connect()` **crea el fichero si no existe**. No hay que hacer nada más para "instalar" la BBDD.
* Paso 2: el `cursor` es el objeto sobre el que ejecutas comandos SQL.
* Paso 3: cada `execute()` es una operación SQL.
* Paso 4: `commit()` es **imprescindible** si has modificado datos. Si lo olvidas, **tus cambios se pierden**.
* Paso 5: cerrar libera el fichero para otros programas.

## 2. Crear la tabla

El SQL para crear una tabla es **`CREATE TABLE nombre (columna1 tipo1, columna2 tipo2, ...)`**. En SQLite, los tipos principales son:

* **`INTEGER`** — números enteros.
* **`REAL`** — números con decimales.
* **`TEXT`** — cadenas.
* **`BLOB`** — datos binarios (raro que lo uses en este curso).

Al declarar una columna se pueden añadir restricciones: **`PRIMARY KEY`** (clave primaria), **`NOT NULL`** (obligatorio), **`UNIQUE`** (único).

In [ ]:
import sqlite3

conexion = sqlite3.connect('universidad.db')
cursor = conexion.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS Estudiantes (
        dni    TEXT PRIMARY KEY,
        nombre TEXT NOT NULL,
        curso  INTEGER,
        nota   REAL
    )
''')

conexion.commit()
print('✅ Tabla Estudiantes creada (o ya existía).')

# Vaciamos por si quedaron datos de una ejecución anterior
cursor.execute('DELETE FROM Estudiantes')
conexion.commit()

El `IF NOT EXISTS` evita error si la tabla ya está creada. Es una buena costumbre.

## 3. Insertar registros

El SQL de inserción es `INSERT INTO tabla VALUES (v1, v2, ...)`. Pero **NUNCA** debes construir el SQL concatenando strings con los datos del usuario:

```python
# ❌ MAL — vulnerable a INYECCIÓN SQL
cursor.execute(f"INSERT INTO Users VALUES ('{nombre}', {edad})")
```

En lugar de eso, usa **placeholders con `?`** y pasa los datos como **tupla**:

```python
# ✅ BIEN — seguro contra inyección
cursor.execute('INSERT INTO Users VALUES (?, ?)', (nombre, edad))
```

SQLite tratará el valor como **dato**, no como código SQL, y no se puede romper por texto malicioso.

In [ ]:
estudiantes = [
    ('50123456A', 'Ana',    1, 8.5),
    ('50234567B', 'Luis',   1, 6.7),
    ('50345678C', 'Marta',  2, 9.2),
    ('50456789D', 'Pedro',  1, 4.5),
    ('50567890E', 'Sofía',  2, 7.8),
    ('50678901F', 'Carlos', 3, 6.0),
]

for est in estudiantes:
    cursor.execute('INSERT INTO Estudiantes VALUES (?, ?, ?, ?)', est)

conexion.commit()  # 🔑 sin esto los cambios NO se guardan
print(f'✅ Insertados {len(estudiantes)} estudiantes.')

## 4. Consultar datos con `SELECT`

El SQL de consulta es `SELECT columnas FROM tabla`. Los resultados se obtienen con `cursor.fetchall()` (todos) o `cursor.fetchone()` (uno).

### 4a) Consulta básica

In [ ]:
cursor.execute('SELECT nombre, nota FROM Estudiantes')

resultados = cursor.fetchall()      # ← lista de tuplas
for nombre, nota in resultados:
    print(f'  {nombre:10} → {nota}')

### 4b) Filtrar con `WHERE`

Las condiciones se ponen tras `WHERE`. **Usa placeholders `?`** también para los valores:

In [ ]:
cursor.execute('SELECT nombre, nota FROM Estudiantes WHERE nota >= ?', (7.0,))
for nombre, nota in cursor.fetchall():
    print(f'  ⭐ {nombre}: {nota}')

### 4c) Funciones de agregación

El poder de SQL: en una sola consulta obtienes media, máximo, mínimo, cuenta, suma...

In [ ]:
cursor.execute('SELECT AVG(nota), MAX(nota), MIN(nota), COUNT(*) FROM Estudiantes')
media, maximo, minimo, total = cursor.fetchone()

print(f'Total:  {total}')
print(f'Media:  {media:.2f}')
print(f'Máx:    {maximo}')
print(f'Mín:    {minimo}')

### 4d) Agrupar con `GROUP BY`

**El poder real de SQL** aparece aquí: agrupar por una columna y calcular estadísticas por grupo. Con Python puro necesitarías 10 líneas de código; con SQL, una:

In [ ]:
cursor.execute('''
    SELECT curso, COUNT(*), AVG(nota), MAX(nota)
    FROM Estudiantes
    GROUP BY curso
    ORDER BY curso
''')

print(f'{"Curso":<8} {"N":>3} {"Media":>7} {"Máx":>5}')
print('-' * 26)
for curso, cuantos, media, maximo in cursor.fetchall():
    print(f'  {curso:<6} {cuantos:>3} {media:>7.2f} {maximo:>5}')

## 5. Modificar y eliminar

Para completar el cuadro, los otros dos comandos SQL básicos:

* **`UPDATE tabla SET col=? WHERE ...`** — modificar registros.
* **`DELETE FROM tabla WHERE ...`** — eliminar registros.

⚠️ **Siempre** pon un `WHERE` en `UPDATE` y `DELETE` — sin él, afectas a **toda la tabla**.

In [ ]:
# Aprobado por decreto: subimos la nota de Pedro a 5.0
cursor.execute('UPDATE Estudiantes SET nota = ? WHERE dni = ?', (5.0, '50456789D'))
conexion.commit()

# Verificamos
cursor.execute('SELECT nombre, nota FROM Estudiantes WHERE dni = ?', ('50456789D',))
print(cursor.fetchone())

# Y eliminamos a Carlos (que ya se ha graduado)
cursor.execute('DELETE FROM Estudiantes WHERE dni = ?', ('50678901F',))
conexion.commit()

# Recuento final
cursor.execute('SELECT COUNT(*) FROM Estudiantes')
print(f'Estudiantes tras el cambio: {cursor.fetchone()[0]}')

## 6. Cerrar la conexión

Es una buena costumbre cerrar la conexión al terminar. Si tu programa se cae con la conexión abierta, el fichero puede quedar bloqueado.

In [ ]:
conexion.close()
print('✅ Conexión cerrada.')

## 🎯 Resumen: la chuleta de SQLite

```python
import sqlite3

# CONECTAR + CURSOR
conexion = sqlite3.connect('mi.db')
cursor = conexion.cursor()

# CREAR TABLA
cursor.execute('CREATE TABLE IF NOT EXISTS T (id INTEGER PRIMARY KEY, ...)')

# INSERTAR
cursor.execute('INSERT INTO T VALUES (?, ?)', (dato1, dato2))
conexion.commit()

# CONSULTAR
cursor.execute('SELECT ... FROM T WHERE ...', (param,))
resultados = cursor.fetchall()

# MODIFICAR / ELIMINAR
cursor.execute('UPDATE T SET ... WHERE ...', (nuevo,))
cursor.execute('DELETE FROM T WHERE ...', (param,))
conexion.commit()

# CERRAR
conexion.close()
```

## 🛠️ Herramienta muy recomendada: DB Browser for SQLite

Instala [**DB Browser for SQLite**](https://sqlitebrowser.org/) (gratis) y ábrelo con el fichero `universidad.db` que acabas de crear. Podrás:

* Ver las tablas y sus datos visualmente.
* Ejecutar consultas SQL desde una interfaz gráfica.
* Editar registros a mano.

Es como tener un Excel al lado mientras programas — muy útil para depurar.

## 🚀 Para reflexionar

* ¿Cómo cargarías los datos de un **CSV** directamente en la tabla `Estudiantes`? (Pista: recorres el CSV y haces un INSERT por cada fila.)
* ¿Y si necesitaras que la tabla tuviera **dos claves** (curso + dni) que juntas fueran únicas? (Pista: `PRIMARY KEY (curso, dni)`.)